In [1]:
# [BLOCK 0] Import Libraries
!pip install -q xgboost

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_log_error
from xgboost import XGBRegressor

In [2]:
# [BLOCK 1] Configuration & Load Dataset
DATA_DIR = Path("/kaggle/input/competitions/playground-series-s4e12") 

TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"
SUBMISSION_FILE = "submission_final.csv" # Lưu thẳng ra thư mục /kaggle/working/ mặc định

TARGET = "Premium Amount"
ID_COL = "id"
SEED = 42

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

FEATURES = [col for col in train_df.columns if col not in [ID_COL, TARGET]]

X_full = train_df[FEATURES].copy()
y_full = train_df[TARGET].copy()
X_test = test_df[FEATURES].copy()

In [3]:
# [BLOCK 2] FEATURE ENGINEERING & PREPROCESSOR
def create_features(X_data):
    X_data = X_data.copy()
    
    if "Policy Start Date" in X_data.columns:
        date = pd.to_datetime(X_data["Policy Start Date"], errors="coerce")
        X_data["Policy Start Year"] = date.dt.year
        X_data["Policy Start Month"] = date.dt.month
        X_data["Policy Start Day"] = date.dt.day
        X_data["Policy Start DayOfWeek"] = date.dt.dayofweek
        X_data = X_data.drop(columns=["Policy Start Date"])
        
    if "Annual Income" in X_data.columns and "Number of Dependents" in X_data.columns:
        X_data['Income_per_Dependent'] = X_data['Annual Income'] / (X_data['Number of Dependents'] + 1)
        
    if "Vehicle Age" in X_data.columns and "Age" in X_data.columns:
        X_data['Vehicle_Age_Ratio'] = X_data['Vehicle Age'] / (X_data['Age'] + 1)
        
    if "Previous Claims" in X_data.columns and "Insurance Duration" in X_data.columns:
        X_data['Claims_per_Year'] = X_data['Previous Claims'] / (X_data['Insurance Duration'] + 0.1)
        
    if "Health Score" in X_data.columns and "Age" in X_data.columns:
        X_data['Health_Age_Index'] = X_data['Health Score'] * X_data['Age']
        
    return X_data

X_full = create_features(X_full)
X_test = create_features(X_test)

In [4]:
# [BLOCK 3] PREPROCESSOR
cat_features = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
num_features = [col for col in X_full.columns if col not in cat_features]

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_features),
    ("cat", categorical_pipeline, cat_features)
])

In [6]:
# [BLOCK 4] VALIDATION & XGBOOST
target_bins = pd.qcut(y_full, q=20, labels=False, duplicates="drop")
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=target_bins
)

model_val = XGBRegressor(
    n_estimators=1000, 
    max_depth=8, 
    learning_rate=0.01, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    min_child_weight=3, 
    objective="reg:squarederror", 
    random_state=SEED,
    tree_method='hist',
    device='cuda'
)

pipeline_val = Pipeline([("preprocessor", preprocessor), ("model", model_val)])
pipeline_val.fit(X_train, np.log1p(y_train))

val_pred = pipeline_val.predict(X_val)
val_pred = np.expm1(val_pred)
val_pred = np.clip(val_pred, 0, None)

rmsle = np.sqrt(mean_squared_log_error(y_val, val_pred))
print(f"ĐIỂM RMSLE TRÊN TẬP VALIDATION: {rmsle:.6f}")

/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [08:24:37] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


ĐIỂM RMSLE TRÊN TẬP VALIDATION: 1.046068


In [9]:
# [BLOCK 5] FULL TRAINING
final_model = XGBRegressor(
    n_estimators=1000, 
    max_depth=8, 
    learning_rate=0.01, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    min_child_weight=3, 
    objective="reg:squarederror", 
    random_state=SEED,
    tree_method='hist', 
    device='cuda'       
)

final_pipeline = Pipeline([("preprocessor", preprocessor), ("model", final_model)])
final_pipeline.fit(X_full, np.log1p(y_full))

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Annual Income',
                                                   'Number of Dependents',
                                                   'Health Score',
                                                   'Previous Claims',
                                                   'Vehicle Age',
                                                   'Credit Score',
                                                   'Insurance Duration',
                                                   'Policy Start Year',
                                                   'Policy Start Month',
                                                   'Policy Start Day',
                                                   'Policy Start DayOfWeek',
                                                   'Income_per_Depend...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.01,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=8, max_leaves=None, min_child_weight=3,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=1000,
                              n_jobs=None, num_parallel_tree=None, ...))])

In [10]:
# [BLOCK 6] Export to CSV
test_pred = final_pipeline.predict(X_test)
test_pred = np.expm1(test_pred)
test_pred = np.clip(test_pred, 0, None)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred
})

submission.to_csv(SUBMISSION_FILE, index=False)